# ReviewLens — Turkish BERT Fine-Tuning
**Runtime → Change runtime type → T4 GPU**

In [ ]:
# 1. Install dependencies
!pip install transformers torch datasets -q

In [ ]:
# 2. Upload your training data
# Run prepare_data.py locally first, then upload training/data/ folder
from google.colab import files
import os
os.makedirs('training/data', exist_ok=True)
print('Upload train.json, val.json, test.json from your training/data/ folder')
uploaded = files.upload()

In [ ]:
import shutil
for fname in uploaded:
    shutil.move(fname, f'training/data/{fname}')
print('Files moved:', os.listdir('training/data'))

In [ ]:
# 3. Fine-tune
import json, time, torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

MODEL_NAME = 'savasy/bert-base-turkish-sentiment-cased'
EPOCHS = 3
BATCH_SIZE = 16  # GPU can handle larger batches
MAX_LENGTH = 128
LR = 2e-5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class ReviewDataset(Dataset):
    def __init__(self, path, tokenizer):
        with open(path) as f:
            self.data = json.load(f)
        self.tok = tokenizer
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        enc = self.tok(self.data[i]['text'], truncation=True, max_length=MAX_LENGTH, padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(), 'attention_mask': enc['attention_mask'].squeeze(), 'labels': torch.tensor(self.data[i]['label'], dtype=torch.long)}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

train_loader = DataLoader(ReviewDataset('training/data/train.json', tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(ReviewDataset('training/data/val.json',   tokenizer), batch_size=BATCH_SIZE)
test_loader  = DataLoader(ReviewDataset('training/data/test.json',  tokenizer), batch_size=BATCH_SIZE)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, int(len(train_loader)*EPOCHS*0.1), len(train_loader)*EPOCHS)

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for b in loader:
            out = model(b['input_ids'].to(device), b['attention_mask'].to(device))
            preds = out.logits.argmax(-1)
            correct += (preds == b['labels'].to(device)).sum().item()
            total += b['labels'].size(0)
    return correct / total

best_acc = 0
for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        out = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), labels=batch['labels'].to(device))
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total_loss += out.loss.item()
    val_acc = evaluate(model, val_loader)
    print(f'Epoch {epoch} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}')
    if val_acc > best_acc:
        best_acc = val_acc
        model.save_pretrained('finetuned_bert')
        tokenizer.save_pretrained('finetuned_bert')
        print(f'  Best model saved!')

test_acc = evaluate(model, test_loader)
print(f'\nTest Accuracy: {test_acc:.4f}')

In [ ]:
# 4. Download fine-tuned model
import shutil
shutil.make_archive('finetuned_bert', 'zip', 'finetuned_bert')
files.download('finetuned_bert.zip')
print('Download complete! Extract to reviewlens/models/finetuned_bert/')